# Recomendação de Filmes com Aprendizado por Reforço

Este notebook implementa um sistema de recomendação baseado em **Reinforcement Learning (RL)** com **três agentes DQN cooperativos**, treinados com a biblioteca [Tianshou](https://tianshou.org).

## Arquitetura geral

```
Estado do usuário
       │
       ▼
  ┌─────────┐     gênero     ┌─────────┐   diversidade   ┌─────────┐   país+idioma
  │ Agente 1│ ─────────────► │ Agente 2│ ──────────────► │ Agente 3│ ──────────────► filme
  └─────────┘                └─────────┘                 └─────────┘
```

| Agente | Decide | Recompensa |
|--------|--------|------------|
| 1 | Gênero do próximo filme | Preferência histórica + diversidade vs. últimos 10 filmes do usuário |
| 2 | Grupo de diversidade (gênero/raça do diretor) | Distância do centróide dos últimos 100 recomendados |
| 3 | País de origem + idioma | Distância geográfica/linguística do histórico |

## Arquivos necessários

| Arquivo | Descrição |
|---------|----------|
| `movie_encoding.tsv` | Features binárias de cada filme (gênero, diversidade, idioma, país) |
| `user_ratings.csv` | Histórico de avaliações dos usuários (USERID, MOVIEID, RATING) |
| `recommendations.tsv` | Histórico de recomendações do sistema (opcional) |

## 1. Instalação de Dependências

In [ ]:
# Versões fixas para garantir compatibilidade no Colab
!pip install "tianshou==0.5.3" "gym==0.26.2" torch numpy --quiet

import tianshou, gym, torch, numpy
print(f'tianshou {tianshou.__version__}  |  gym {gym.__version__}  |  torch {torch.__version__}')

## 2. Upload dos Arquivos

Faça upload dos três arquivos gerados pelo projeto `rs`:
- `data/movie_encoding.tsv`
- `data/user_ratings.csv` *(exportado do Oracle)*
- `data/recommendations.tsv` *(opcional — exportado da tabela RECOMMENDATION)*

Se preferir usar Google Drive, comente o bloco de upload e descomente o bloco do Drive.

In [ ]:
import os

# ── Opção A: upload direto ────────────────────────────────────────────────────
#from google.colab import files
#uploaded = files.upload()   # selecione os arquivos no diálogo

# ── Opção B: Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/data/rec_rl'

#BASE = '/content/'

MOVIE_ENC_PATH = os.path.join(BASE, 'movie_encoding.tsv')
RATINGS_PATH   = os.path.join(BASE, 'user_ratings.csv')
RECS_PATH      = os.path.join(BASE, 'recommendations.tsv')   # opcional

print('movie_encoding.tsv:', os.path.exists(MOVIE_ENC_PATH))
print('user_ratings.csv:  ', os.path.exists(RATINGS_PATH))
print('recommendations.tsv:', os.path.exists(RECS_PATH), '(opcional)')

## 3. Exploração dos Dados

Antes de treinar, vamos entender a estrutura dos arquivos e a distribuição das features.

O arquivo `movie_encoding.tsv` tem uma linha por filme e colunas binárias (0/1) organizadas em quatro blocos:

```
movieid | imdbid | title | genre_action | genre_drama | ... | div_male_white | ... | lang_english | ... | country_usa | ...
```

In [ ]:
import pandas as pd

enc = pd.read_csv(MOVIE_ENC_PATH, sep='\t', low_memory=False)
print(f'Filmes: {len(enc):,}')
print(f'Colunas totais: {len(enc.columns)}')
print()

# Identifica os grupos de colunas
genre_cols   = [c for c in enc.columns if c.startswith('genre_')]
div_cols     = [c for c in enc.columns if c.startswith('div_')]
lang_cols    = [c for c in enc.columns if c.startswith('lang_')]
country_cols = [c for c in enc.columns if c.startswith('country_')]

print(f'Gêneros    : {len(genre_cols)} colunas')
print(f'Diversidade: {len(div_cols)} colunas')
print(f'Idiomas    : {len(lang_cols)} colunas (top-19 + outros)')
print(f'Países     : {len(country_cols)} colunas (top-29 + outros)')
print(f'Feature dim: {len(genre_cols) + len(div_cols) + len(lang_cols) + len(country_cols)}')

enc.head(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Distribuição de filmes por feature', fontsize=14)

def plot_counts(ax, cols, prefix, title, top=15):
    counts = enc[cols].sum().sort_values(ascending=False).head(top)
    counts.index = counts.index.str.replace(prefix, '', regex=False)
    counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('N° de filmes')
    ax.invert_yaxis()

plot_counts(axes[0,0], genre_cols,   'genre_',   'Gêneros')
plot_counts(axes[0,1], div_cols,     'div_',     'Diversidade (diretor)')
plot_counts(axes[1,0], lang_cols,    'lang_',    'Idiomas')
plot_counts(axes[1,1], country_cols, 'country_', 'Países')

plt.tight_layout()
plt.show()

In [ ]:
# Avaliações dos usuários
ratings_df = pd.read_csv(RATINGS_PATH)
# Normaliza nomes de colunas para lowercase
ratings_df.columns = ratings_df.columns.str.upper()

print(f'Avaliações: {len(ratings_df):,}')
print(f'Usuários  : {ratings_df["USERID"].nunique():,}')
print(f'Filmes    : {ratings_df["MOVIEID"].nunique():,}')
print(f'Rating médio: {ratings_df["RATING"].mean():.2f}')

ratings_df['RATING'].hist(bins=10, figsize=(8,3), color='steelblue', edgecolor='white')
plt.title('Distribuição de ratings')
plt.xlabel('Rating')
plt.show()

## 4. Carregamento de Dados (`FileDataLoader`)

Substituímos a conexão Oracle por leitura direta dos arquivos TSV/CSV.

O `FileDataLoader` reconstrói as mesmas estruturas do `DataLoader` original:
- `movies`: dict com features de cada filme
- `genres`, `languages`, `countries`: listas de categorias
- `train_user_ids` / `test_user_ids`: split 80/20

In [ ]:
import csv
import random
from collections import defaultdict
from typing import Optional

import numpy as np

HISTORY_USER_SIZE  = 10
HISTORY_SYS_SIZE   = 100
N_DIVERSITY_GROUPS = 9

DIVERSITY_LABELS = [
    'Fem+NoWhite', 'Fem+White', 'Fem+Mix',
    'Male+NoWhite', 'Male+White', 'Male+Mix',
    'Mix+NoWhite', 'Mix+White', 'Mix+Unknown',
]

class FileDataLoader:
    """
    Carrega dados de features e avaliações a partir de arquivos locais,
    sem necessidade de conexão com banco de dados.
    """

    def __init__(self, enc_path: str, ratings_path: str, recs_path: Optional[str] = None):
        self._load_encoding(enc_path)
        self._load_ratings(ratings_path)
        self._recs_path = recs_path
        self._split_users()
        self._print_summary()

    # ── Carregamento do movie_encoding.tsv ────────────────────────────────────

    def _load_encoding(self, path: str):
        print('Carregando movie_encoding.tsv ...')
        with open(path, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f, delimiter='\t')
            header = reader.fieldnames

            # Identifica colunas de cada grupo
            genre_cols   = [c for c in header if c.startswith('genre_')]
            div_cols     = [c for c in header if c.startswith('div_')]
            lang_cols    = [c for c in header if c.startswith('lang_') and c != 'lang_other']
            country_cols = [c for c in header if c.startswith('country_') and c != 'country_other']

            # Vocabulários de categorias
            self.genres       = [c[len('genre_'):]   for c in genre_cols]
            self.languages    = [c[len('lang_'):]    for c in lang_cols]
            self.countries    = [c[len('country_'):] for c in country_cols]
            self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
            self.lang_to_idx  = {l: i for i, l in enumerate(self.languages)}
            self.country_to_idx = {c: i for i, c in enumerate(self.countries)}
            self.n_genres     = len(self.genres)

            # Lê cada linha e reconstrói as features
            self.movies: dict[int, dict] = {}
            for row in reader:
                try:
                    mid = int(row['movieid'])
                except (ValueError, TypeError):
                    continue

                m_genres    = {self.genres[i]    for i, c in enumerate(genre_cols)    if row.get(c) == '1'}
                m_langs     = {self.languages[i] for i, c in enumerate(lang_cols)     if row.get(c) == '1'}
                m_countries = {self.countries[i] for i, c in enumerate(country_cols)  if row.get(c) == '1'}
                has_other_lang    = row.get('lang_other', '0') == '1'
                has_other_country = row.get('country_other', '0') == '1'

                # Grupo de diversidade: índice do div_ col que vale 1
                div_group = N_DIVERSITY_GROUPS - 1  # default: Mix+Unknown
                for i, c in enumerate(div_cols):
                    if row.get(c) == '1':
                        div_group = i
                        break

                self.movies[mid] = {
                    'title':             row.get('title', ''),
                    'imdbid':            row.get('imdbid', ''),
                    'genres':            m_genres,
                    'languages':         m_langs,
                    'countries':         m_countries,
                    'has_other_lang':    has_other_lang,
                    'has_other_country': has_other_country,
                    'diversity':         div_group,
                }

        self.movie_ids = list(self.movies.keys())

        # Combinações país × idioma (top 50 por frequência)
        from collections import Counter
        cl_counter = Counter()
        for feat in self.movies.values():
            for c in feat['countries']:
                for l in feat['languages']:
                    cl_counter[(c, l)] += 1
        self.country_lang_combos = [pair for pair, _ in cl_counter.most_common(50)]
        self.cl_to_idx = {cl: i for i, cl in enumerate(self.country_lang_combos)}
        self.n_country_lang = len(self.country_lang_combos)

    # ── Carregamento do user_ratings.csv ─────────────────────────────────────

    def _load_ratings(self, path: str):
        print('Carregando user_ratings.csv ...')
        self._all_ratings: dict[int, dict[int, float]] = defaultdict(dict)
        with open(path, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            # Normaliza nomes de colunas
            for row in reader:
                row = {k.upper(): v for k, v in row.items()}
                try:
                    uid = int(row['USERID'])
                    mid = int(row['MOVIEID'])
                    rat = float(row['RATING'])
                except (ValueError, TypeError, KeyError):
                    continue
                self._all_ratings[uid][mid] = rat
        self.user_ids = list(self._all_ratings.keys())

    # ── Split treino / teste 80/20 ────────────────────────────────────────────

    def _split_users(self):
        shuffled = self.user_ids.copy()
        random.shuffle(shuffled)
        split = int(len(shuffled) * 0.8)
        self.train_user_ids = shuffled[:split]
        self.test_user_ids  = shuffled[split:]

    def _print_summary(self):
        print(f'  {len(self.movie_ids):,} filmes  |  {self.n_genres} gêneros  |  '
              f'{len(self.languages)} idiomas  |  {len(self.countries)} países')
        print(f'  {len(self.user_ids):,} usuários '
              f'(treino={len(self.train_user_ids):,}, teste={len(self.test_user_ids):,})')
        print(f'  {self.n_country_lang} combos país×idioma')

    # ── Acesso por usuário ────────────────────────────────────────────────────

    def load_user_ratings(self, user_id: int) -> dict[int, float]:
        return dict(self._all_ratings.get(user_id, {}))

    def load_system_history(self, user_id: int) -> list[int]:
        """Retorna últimos HISTORY_SYS_SIZE filmes recomendados ao usuário."""
        if not self._recs_path or not os.path.exists(self._recs_path):
            return []
        result = []
        with open(self._recs_path, newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f, delimiter='\t'):
                row = {k.upper(): v for k, v in row.items()}
                try:
                    if int(row.get('USERID', -1)) == user_id:
                        result.append(int(row.get('ITEM_ID') or row.get('MOVIEID')))
                except (ValueError, TypeError):
                    continue
        return result[-HISTORY_SYS_SIZE:]


# Carrega os dados
data = FileDataLoader(
    enc_path=MOVIE_ENC_PATH,
    ratings_path=RATINGS_PATH,
    recs_path=RECS_PATH if os.path.exists(RECS_PATH) else None,
)
print('\nPronto!')

## 5. Feature Encoding

O `FeatureEncoder` converte as features de cada filme em um **vetor numérico de dimensão fixa**,
concatenando quatro blocos:

```
┌──────────────┬────────────┬────────────────────┬──────────────────────┐
│ genre (multi)│ div (one)  │  lang (multi+other) │ country (multi+other)│
│   ~20 dims   │   9 dims   │      20 dims         │       30 dims         │
└──────────────┴────────────┴────────────────────┴──────────────────────┘
                           ≈ 79 dims total
```

O estado de cada agente é a **média** dos vetores dos filmes no histórico do usuário,
o que gera um "perfil agregado" do gosto do usuário.

In [ ]:
class FeatureEncoder:

    def __init__(self, data: FileDataLoader):
        self.data = data
        self.feature_dim = (
            data.n_genres
            + N_DIVERSITY_GROUPS
            + len(data.languages) + 1    # +1 bucket outros
            + len(data.countries) + 1    # +1 bucket outros
        )
        print(f'feature_dim = {self.feature_dim}')

    def encode_movie(self, movie_id: int) -> np.ndarray:
        vec  = np.zeros(self.feature_dim, dtype=np.float32)
        feat = self.data.movies.get(movie_id, {})
        off  = 0

        # Bloco 1 – gêneros (multi-hot)
        for g in feat.get('genres', []):
            if g in self.data.genre_to_idx:
                vec[off + self.data.genre_to_idx[g]] = 1.0
        off += self.data.n_genres

        # Bloco 2 – diversidade (one-hot)
        dg = feat.get('diversity', N_DIVERSITY_GROUPS - 1)
        vec[off + min(dg, N_DIVERSITY_GROUPS - 1)] = 1.0
        off += N_DIVERSITY_GROUPS

        # Bloco 3 – idiomas (multi-hot + bucket outros)
        for l in feat.get('languages', []):
            idx = self.data.lang_to_idx.get(l, len(self.data.languages))  # outros
            vec[off + idx] = 1.0
        if feat.get('has_other_lang'):
            vec[off + len(self.data.languages)] = 1.0
        off += len(self.data.languages) + 1

        # Bloco 4 – países (multi-hot + bucket outros)
        for c in feat.get('countries', []):
            idx = self.data.country_to_idx.get(c, len(self.data.countries))  # outros
            vec[off + idx] = 1.0
        if feat.get('has_other_country'):
            vec[off + len(self.data.countries)] = 1.0

        return vec

    def encode_movie_list(self, movie_ids: list) -> np.ndarray:
        """Média dos vetores de uma lista de filmes."""
        vecs = [self.encode_movie(m) for m in movie_ids if m in self.data.movies]
        return np.mean(vecs, axis=0).astype(np.float32) if vecs \
               else np.zeros(self.feature_dim, dtype=np.float32)

    def user_genre_preferences(self, ratings: dict) -> np.ndarray:
        """Vetor (n_genres,) com rating médio do usuário por gênero."""
        totals = np.zeros(self.data.n_genres, dtype=np.float32)
        counts = np.zeros(self.data.n_genres, dtype=np.float32)
        for mid, rating in ratings.items():
            for g in self.data.movies.get(mid, {}).get('genres', []):
                if g in self.data.genre_to_idx:
                    idx = self.data.genre_to_idx[g]
                    totals[idx] += rating
                    counts[idx] += 1.0
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where(counts > 0, totals / counts, 0.0).astype(np.float32)


encoder = FeatureEncoder(data)

# Exemplo: vetor de features do filme 1 (Toy Story)
vec = encoder.encode_movie(1)
print(f'Vetor Toy Story: shape={vec.shape}, não-zeros={int((vec>0).sum())}')
print(f'Gêneros ativos: {data.movies[1]["genres"]}')

## 6. Ambientes Gymnasium

Cada agente tem seu próprio ambiente. Todos compartilham o mesmo estado base:

```
obs_base = [ preferência por gênero (n_genres) ]
         + [ média features últimos 10 filmes do usuário (feature_dim) ]
         + [ média features últimos 100 recomendados (feature_dim) ]
```

Agentes 2 e 3 recebem adicionalmente as escolhas dos agentes anteriores concatenadas ao estado.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from functools import partial


class BaseMovieEnv(gym.Env):
    """Estado compartilhado entre os três agentes."""

    def __init__(self, data: FileDataLoader, encoder: FeatureEncoder,
                 user_ids: Optional[list] = None):
        super().__init__()
        self.data      = data
        self.encoder   = encoder
        self._user_ids = user_ids if user_ids is not None else data.user_ids
        self.base_obs_dim = data.n_genres + 2 * encoder.feature_dim
        self._user_id  = None
        self._ratings  = {}
        self._user_hist = []
        self._sys_hist  = []

    def _base_obs(self) -> np.ndarray:
        pref   = self.encoder.user_genre_preferences(self._ratings)
        u_hist = self.encoder.encode_movie_list(self._user_hist)
        s_hist = self.encoder.encode_movie_list(self._sys_hist)
        return np.concatenate([pref, u_hist, s_hist]).astype(np.float32)

    def _sample_episode(self):
        self._user_id   = random.choice(self._user_ids)
        self._ratings   = self.data.load_user_ratings(self._user_id)
        self._user_hist = list(self._ratings.keys())[-HISTORY_USER_SIZE:]
        self._sys_hist  = self.data.load_system_history(self._user_id)


# ── Agente 1 — Gênero ─────────────────────────────────────────────────────────
class Agent1GenreEnv(BaseMovieEnv):
    """
    Ação: índice de gênero.
    Recompensa: α × preferência histórica + (1-α) × diversidade vs. últimos 10 filmes.
    """
    ALPHA = 0.6

    def __init__(self, data, encoder, user_ids=None):
        super().__init__(data, encoder, user_ids)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(self.base_obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Discrete(data.n_genres)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._sample_episode()
        return self._base_obs(), {}

    def step(self, action: int):
        genre  = self.data.genres[action]
        reward = self._reward(genre)
        return self._base_obs(), reward, True, False, {'genre': genre}

    def _reward(self, genre: str) -> float:
        idx  = self.data.genre_to_idx.get(genre, -1)
        pref = self.encoder.user_genre_preferences(self._ratings)
        pref_score = float(pref[idx]) / 5.0 if idx >= 0 else 0.0
        if not self._user_hist:
            div_score = 1.0
        else:
            presence  = [1.0 if genre in self.data.movies.get(m, {}).get('genres', []) else 0.0
                         for m in self._user_hist]
            div_score = 1.0 - sum(presence) / len(presence)
        return self.ALPHA * pref_score + (1.0 - self.ALPHA) * div_score


# ── Agente 2 — Diversidade ────────────────────────────────────────────────────
class Agent2DiversityEnv(BaseMovieEnv):
    """
    Ação: índice do grupo de diversidade (0-8).
    Recompensa: distância do grupo escolhido ao centróide dos últimos 100 recomendados.
    """
    def __init__(self, data, encoder, user_ids=None):
        super().__init__(data, encoder, user_ids)
        obs_dim = self.base_obs_dim + data.n_genres
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Discrete(N_DIVERSITY_GROUPS)
        self._genre_vec        = np.zeros(data.n_genres, dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._sample_episode()
        g_idx = random.randrange(self.data.n_genres)
        self._genre_vec = np.zeros(self.data.n_genres, dtype=np.float32)
        self._genre_vec[g_idx] = 1.0
        return np.concatenate([self._base_obs(), self._genre_vec]), {}

    def step(self, action: int):
        obs    = np.concatenate([self._base_obs(), self._genre_vec])
        reward = self._reward(action)
        return obs, reward, True, False, {'diversity_group': action}

    def _reward(self, group_idx: int) -> float:
        if not self._sys_hist:
            return 0.5
        chosen = np.zeros(N_DIVERSITY_GROUPS, dtype=np.float32)
        chosen[group_idx] = 1.0
        hist_vecs = []
        for mid in self._sys_hist:
            dg = self.data.movies.get(mid, {}).get('diversity', N_DIVERSITY_GROUPS - 1)
            v  = np.zeros(N_DIVERSITY_GROUPS, dtype=np.float32)
            v[min(dg, N_DIVERSITY_GROUPS - 1)] = 1.0
            hist_vecs.append(v)
        centroid = np.mean(hist_vecs, axis=0)
        return min(float(np.linalg.norm(chosen - centroid)) / np.sqrt(2), 1.0)


# ── Agente 3 — País + Idioma ──────────────────────────────────────────────────
class Agent3GeoLangEnv(BaseMovieEnv):
    """
    Ação: índice da combinação (país, idioma).
    Recompensa: distância geográfica/linguística do centróide dos últimos 100 recomendados.
    """
    def __init__(self, data, encoder, user_ids=None):
        super().__init__(data, encoder, user_ids)
        obs_dim = self.base_obs_dim + data.n_genres + N_DIVERSITY_GROUPS
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Discrete(data.n_country_lang)
        self._genre_vec = np.zeros(data.n_genres, dtype=np.float32)
        self._div_vec   = np.zeros(N_DIVERSITY_GROUPS, dtype=np.float32)

    def _obs(self):
        return np.concatenate([self._base_obs(), self._genre_vec, self._div_vec])

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._sample_episode()
        self._genre_vec = np.zeros(self.data.n_genres, dtype=np.float32)
        self._genre_vec[random.randrange(self.data.n_genres)] = 1.0
        self._div_vec = np.zeros(N_DIVERSITY_GROUPS, dtype=np.float32)
        self._div_vec[random.randrange(N_DIVERSITY_GROUPS)] = 1.0
        return self._obs(), {}

    def step(self, action: int):
        reward = self._reward(action)
        return self._obs(), reward, True, False, {'country_lang_idx': action}

    def _reward(self, combo_idx: int) -> float:
        if not self._sys_hist:
            return 0.5
        n_cl   = self.data.n_country_lang
        chosen = np.zeros(n_cl, dtype=np.float32)
        chosen[combo_idx] = 1.0
        hist_vecs = []
        for mid in self._sys_hist:
            v = np.zeros(n_cl, dtype=np.float32)
            m = self.data.movies.get(mid, {})
            for c in m.get('countries', []):
                for l in m.get('languages', []):
                    idx = self.data.cl_to_idx.get((c, l))
                    if idx is not None:
                        v[idx] = 1.0
            hist_vecs.append(v)
        centroid = np.mean(hist_vecs, axis=0)
        return min(float(np.linalg.norm(chosen - centroid)) / np.sqrt(n_cl), 1.0)


# Verificação das dimensões
e1 = Agent1GenreEnv(data, encoder, data.train_user_ids)
e2 = Agent2DiversityEnv(data, encoder, data.train_user_ids)
e3 = Agent3GeoLangEnv(data, encoder, data.train_user_ids)

print(f'Agente 1 — obs: {e1.observation_space.shape[0]}  ações: {e1.action_space.n}')
print(f'Agente 2 — obs: {e2.observation_space.shape[0]}  ações: {e2.action_space.n}')
print(f'Agente 3 — obs: {e3.observation_space.shape[0]}  ações: {e3.action_space.n}')
e1.close(); e2.close(); e3.close()

## 7. Políticas DQN (Tianshou)

Cada agente usa um **DQN (Deep Q-Network)** com uma rede MLP de 3 camadas.

### Como o DQN funciona

A rede neural aproxima a função Q(s, a): o valor esperado de recompensa ao tomar a ação `a` no estado `s`.

```
estado (obs_dim) → [256] → [256] → [128] → Q(s, a) para cada ação
```

### Estratégia ε-greedy

- **Exploração** (ε alto): ação aleatória — descobre combinações novas
- **Exploitation** (ε baixo): ação com maior Q — usa o que já aprendeu
- ε decai **linearmente por step**: 0.5 → 0.05 ao longo do treino

In [ ]:
import torch
from tianshou.data import Batch, Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from tianshou.utils.net.common import Net


def build_dqn(obs_dim: int, action_dim: int, lr: float = 1e-3) -> DQNPolicy:
    """Cria política DQN com MLP 256→256→128."""
    net = Net(
        state_shape=obs_dim,
        action_shape=action_dim,
        hidden_sizes=[256, 256, 128],
        device='cpu',
    )
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    return DQNPolicy(
        model=net,
        optim=optimizer,
        action_space=spaces.Discrete(action_dim),
        discount_factor=0.99,
        estimation_step=1,
        target_update_freq=200,
    )

print('Função build_dqn definida.')

## 8. Treinamento

Cada agente é treinado **independentemente** no seu próprio ambiente.

O `OffpolicyTrainer` do Tianshou gerencia:
- **Coleta** de experiências com o `train_collector` (usuários de treino)
- **Atualização** da rede a cada `step_per_collect` transições
- **Avaliação** a cada época com o `test_collector` (usuários de teste)

> ⚠️ O treino pode demorar dependendo do número de usuários. Reduza `n_epoch` para testes rápidos.

In [ ]:
def train_agent(env_cls, data, encoder, label='',
                n_envs=4, buffer_size=20_000,
                n_epoch=5, step_per_epoch=2_000,
                step_per_collect=100, batch_size=256, lr=1e-3):

    print(f'\n{"─"*60}')
    print(f'Treinando {label} ...')

    make_train = partial(env_cls, data, encoder, data.train_user_ids)
    make_test  = partial(env_cls, data, encoder, data.test_user_ids)
    train_envs = DummyVectorEnv([make_train] * n_envs)
    test_envs  = DummyVectorEnv([make_test])

    ref        = make_train()
    obs_dim    = ref.observation_space.shape[0]
    action_dim = ref.action_space.n
    ref.close()

    policy  = build_dqn(obs_dim, action_dim, lr=lr)
    buffer  = VectorReplayBuffer(buffer_size, buffer_num=n_envs)
    train_c = Collector(policy, train_envs, buffer, exploration_noise=True)
    test_c  = Collector(policy, test_envs)

    train_c.collect(n_step=batch_size * 4)   # pré-preenche o buffer

    total_steps = n_epoch * step_per_epoch

    def train_fn(_epoch, global_step):
        # Decaimento linear de ε por step: 0.5 → 0.05
        eps = max(0.05, 0.5 - 0.45 * global_step / total_steps)
        policy.set_eps(eps)

    def test_fn(_epoch, _step):
        policy.set_eps(0.0)   # greedy na avaliação

    result = OffpolicyTrainer(
        policy=policy,
        train_collector=train_c,
        test_collector=test_c,
        max_epoch=n_epoch,
        step_per_epoch=step_per_epoch,
        step_per_collect=step_per_collect,
        episode_per_test=20,
        batch_size=batch_size,
        train_fn=train_fn,
        test_fn=test_fn,
        verbose=True,
    ).run()

    print(f'  Concluído. Melhor recompensa: {result["best_reward"]:.4f}')
    train_envs.close()
    test_envs.close()
    return policy

print('Função train_agent definida.')

In [ ]:
# ── Parâmetros de treino ──────────────────────────────────────────────────────
# Reduza N_EPOCH para teste rápido (ex: 2).
# Para resultados melhores, use N_EPOCH=10 ou mais.
N_EPOCH = 5

policy1 = train_agent(Agent1GenreEnv,     data, encoder, label='Agente 1 (Gênero)',       n_epoch=N_EPOCH)
policy2 = train_agent(Agent2DiversityEnv, data, encoder, label='Agente 2 (Diversidade)',  n_epoch=N_EPOCH)
policy3 = train_agent(Agent3GeoLangEnv,   data, encoder, label='Agente 3 (País+Idioma)',  n_epoch=N_EPOCH)

## 9. Geração de Recomendações

Com os três agentes treinados, executamos a **inferência cooperativa**:

1. Agente 1 escolhe o gênero
2. Agente 2 escolhe o grupo de diversidade (condicionado ao gênero)
3. Agente 3 escolhe o país + idioma
4. O sistema busca um filme não visto que satisfaça todas as restrições,
   relaxando progressivamente se não houver candidatos

In [ ]:
def agent_act(policy: DQNPolicy, obs: np.ndarray) -> int:
    policy.set_eps(0.0)
    with torch.no_grad():
        obs_t  = torch.tensor(obs[None], dtype=torch.float32)
        result = policy(Batch(obs=obs_t, info={}))
    return int(result.act[0])


def find_movie(data, seen, genre=None, diversity=None, country=None, language=None):
    def match(mid, g, d, c, l):
        f = data.movies.get(mid, {})
        return (
            mid not in seen
            and (g is None or g in f.get('genres', []))
            and (d is None or f.get('diversity') == d)
            and (c is None or c in f.get('countries', []))
            and (l is None or l in f.get('languages', []))
        )
    for relaxed in [
        (genre, diversity, country, language),
        (genre, None, country, None),
        (genre, None, None, None),
        (None, None, None, None),
    ]:
        pool = [m for m in data.movie_ids if match(m, *relaxed)]
        if pool:
            return random.choice(pool)
    return None


def recommend(user_id: int, n: int = 10) -> list:
    ratings   = data.load_user_ratings(user_id)
    user_hist = list(ratings.keys())[-HISTORY_USER_SIZE:]
    sys_hist  = data.load_system_history(user_id)
    seen      = set(ratings.keys())
    recs      = []

    for _ in range(n):
        pref     = encoder.user_genre_preferences(ratings)
        base_obs = np.concatenate([pref,
                                   encoder.encode_movie_list(user_hist),
                                   encoder.encode_movie_list(sys_hist)])

        # Agente 1 → gênero
        act1  = agent_act(policy1, base_obs)
        genre = data.genres[act1]
        g_vec = np.zeros(data.n_genres, dtype=np.float32); g_vec[act1] = 1.0

        # Agente 2 → diversidade
        act2  = agent_act(policy2, np.concatenate([base_obs, g_vec]))
        d_vec = np.zeros(N_DIVERSITY_GROUPS, dtype=np.float32); d_vec[act2] = 1.0

        # Agente 3 → país + idioma
        act3              = agent_act(policy3, np.concatenate([base_obs, g_vec, d_vec]))
        country, language = data.country_lang_combos[act3]

        mid = find_movie(data, seen, genre=genre, diversity=act2,
                         country=country, language=language)
        if mid is None:
            continue

        seen.add(mid)
        sys_hist = ([mid] + sys_hist)[:HISTORY_SYS_SIZE]
        feat = data.movies.get(mid, {})
        recs.append({
            'movie_id':    mid,
            'title':       feat.get('title', ''),
            'genre':       genre,
            'diversity':   DIVERSITY_LABELS[act2],
            'country':     country,
            'language':    language,
            'all_genres':  list(feat.get('genres', [])),
        })
    return recs


print('Funções de recomendação definidas.')

## 10. Resultados

In [ ]:
# Escolha um usuário para recomendar
USER_ID = data.user_ids[0]   # altere para qualquer ID válido
N_RECS  = 10

user_ratings = data.load_user_ratings(USER_ID)
print(f'Usuário {USER_ID}: {len(user_ratings)} filmes avaliados, '
      f'rating médio = {sum(user_ratings.values())/len(user_ratings):.2f}')
print()

recs = recommend(USER_ID, n=N_RECS)

print(f'{'#':>3}  {"movie_id":>8}  {"Título":<35}  {"Gênero":<18}  {"País":<6}  {"Idioma":<12}  Diversidade')
print('─' * 105)
for i, r in enumerate(recs, 1):
    print(f"{i:>3}. {r['movie_id']:>8}  {r['title']:<35}  "
          f"{r['genre']:<18}  {r['country']:<6}  {r['language']:<12}  {r['diversity']}")

In [ ]:
# Visualização das escolhas dos agentes
from collections import Counter

genres_chosen     = Counter(r['genre']     for r in recs)
diversity_chosen  = Counter(r['diversity'] for r in recs)
countries_chosen  = Counter(r['country']   for r in recs)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Escolhas dos agentes — usuário {USER_ID}', fontsize=13)

for ax, counter, title in zip(axes,
    [genres_chosen, diversity_chosen, countries_chosen],
    ['Agente 1: Gêneros', 'Agente 2: Diversidade', 'Agente 3: Países']):
    if counter:
        keys, vals = zip(*sorted(counter.items(), key=lambda x: -x[1]))
        ax.bar(keys, vals, color='steelblue')
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Compare com o perfil histórico do usuário
pref = encoder.user_genre_preferences(user_ratings)
genres_with_pref = [(g, float(pref[i])) for i, g in enumerate(data.genres) if pref[i] > 0]
genres_with_pref.sort(key=lambda x: -x[1])

print(f'Top gêneros históricos do usuário {USER_ID}:')
for g, avg in genres_with_pref[:8]:
    bar = '█' * int(avg * 6)
    print(f'  {g:<20} avg={avg:.2f}  {bar}')

print()
print('Gêneros recomendados:', dict(genres_chosen))